# 02 · Rules and scoring: why an invoice is LOW or CRITICAL

**Rules decide the facts.** Every check here is a pure Python function in `src/trustagent/rules/`:
same input, same output, no model involved. This notebook runs the real intake path (extraction with the
**real** model, then supplier matching) against the database **inside a transaction that is rolled back at
the end**, so nothing is left behind in the demo data.

The AI review (qualitative signals) comes in phase 3. Here we see what the rules alone conclude.

In [1]:
import pandas as pd

from trustagent.config import get_settings

settings = get_settings()
pd.set_option("display.max_colwidth", 90)
print(f"Generator model: {settings.llm_provider} / {settings.llm_model}  (REAL API calls, not a mock)")

Generator model: gemini / gemini-3.5-flash-lite  (REAL API calls, not a mock)


## 1. Intake all 11 sample invoices (extraction → supplier match → stored invoice)

In [2]:
from trustagent.db.session import get_sessionmaker
from trustagent.extraction.labels import load_labels
from trustagent.workflow.intake import intake_document

session = get_sessionmaker()()  # rolled back at the end of the notebook
samples = [n for n in load_labels() if not n.endswith(".json")]
intakes = {}
for name in samples:
    r = intake_document(session, name, (settings.invoices_dir / name).read_bytes(), submitted_by="notebook")
    intakes[name] = r
pd.DataFrame(
    [
        {
            "file": n,
            "invoice": r.invoice.id,
            "supplier": r.supplier_id,
            "match": r.supplier_match,
            "similarity": round(r.match_confidence, 2),
        }
        for n, r in intakes.items()
    ]
)

,file,invoice,supplier,match,similarity
0,INV-1048-abc-office-solutions.md,INV-1048,SUP-001,MATCHED_EXISTING,1.00
1,INV-1049-metro-cleaning.md,INV-1049,SUP-002,MATCHED_EXISTING,0.85
2,INV-1050-digital-print-co.md,INV-1050,SUP-003,MATCHED_EXISTING,1.00
3,INV-1051-secure-it-solutions.md,INV-1051,SUP-004,NEW_SUPPLIER,0.00
4,INV-1052-quickship-logistics.md,INV-1052,SUP-005,NEW_SUPPLIER,0.00
5,INV-1053-prestige-catering.md,INV-1053,SUP-006,NEW_SUPPLIER,0.00
6,INV-2001-metro-cleaning-LOW.pdf,INV-2001,SUP-002,MATCHED_EXISTING,0.85
7,INV-2002-abc-office-solutions-LOW.pdf,INV-2002,SUP-001,MATCHED_EXISTING,1.00
8,INV-2003-digital-print-co-BANK-CHANGE.pdf,INV-2003,SUP-003,MATCHED_EXISTING,1.00
9,INV-2004-metro-cleaning-DUPLICATE.pdf,INV-2004,SUP-002,MATCHED_EXISTING,0.85


## 2. Run the rule checks and the calculator

`load_check_context` gathers the supplier record, payment history and other invoices from the database,
so the checks themselves stay pure. Each indicator type counts **once**, and the total is capped at 100.

In [3]:
from trustagent.db.repository import load_check_context
from trustagent.rules.checks import run_rule_checks
from trustagent.rules.recommendation import minimum_action
from trustagent.rules.scoring import calculate_risk

results = {}
for name, r in intakes.items():
    ctx = load_check_context(session, r.invoice.id)
    risk = calculate_risk(run_rule_checks(ctx))
    results[name] = (ctx, risk)

pd.DataFrame(
    [
        {
            "invoice": ctx.invoice.id,
            "amount": f"R{ctx.invoice.amount:,.2f}",
            "score": risk.score,
            "level": risk.level.value,
            "minimum action": minimum_action(risk.level, bool(ctx.supplier and ctx.supplier.verified)).value,
            "findings": ", ".join(f"{i.type} ({i.weight})" for i in risk.indicators if i.type != "CONFIRMED_MATCH"),
        }
        for ctx, risk in results.values()
    ]
)

,invoice,amount,score,level,minimum action,findings
0,INV-1048,"R185,000.00",60,HIGH,HOLD_PAYMENT,"BANK_DETAILS_CHANGED (30), UNUSUAL_AMOUNT (20), URGENCY_INDICATOR (10)"
1,INV-1049,"R18,400.00",0,LOW,APPROVE_PAYMENT,
2,INV-1050,"R195,500.00",45,MEDIUM,REQUEST_VERIFICATION,"BANK_DETAILS_CHANGED (30), AMOUNT_EXCEEDS_THRESHOLD (15)"
3,INV-1051,"R106,950.00",30,MEDIUM,REQUEST_VERIFICATION,"SUPPLIER_NOT_VERIFIED (15), AMOUNT_EXCEEDS_THRESHOLD (15)"
4,INV-1052,"R126,500.00",65,HIGH,HOLD_PAYMENT,"SUPPLIER_NOT_VERIFIED (15), ACCOUNT_HOLDER_MISMATCH (25), AMOUNT_EXCEEDS_THRESHOLD (15..."
5,INV-1053,"R215,625.00",30,MEDIUM,REQUEST_VERIFICATION,"SUPPLIER_NOT_VERIFIED (15), AMOUNT_EXCEEDS_THRESHOLD (15)"
6,INV-2001,"R17,250.00",0,LOW,APPROVE_PAYMENT,
7,INV-2002,"R39,146.00",0,LOW,APPROVE_PAYMENT,
8,INV-2003,"R43,585.00",65,HIGH,HOLD_PAYMENT,"BANK_DETAILS_CHANGED (30), EMAIL_DOMAIN_MISMATCH (25), URGENCY_INDICATOR (10)"
9,INV-2004,"R18,400.00",35,MEDIUM,REQUEST_VERIFICATION,DUPLICATE_INVOICE (35)


## 3. Why is INV-1049 LOW?

A LOW score is only convincing if the evidence says *what was checked and found clean*.
Passed checks are recorded as `CONFIRMED_MATCH` with weight 0.

In [4]:
def evidence(name):
    ctx, risk = results[name]
    print(f"{ctx.invoice.id}: {risk.score}/100 → {risk.level.value}\n")
    return pd.DataFrame(
        [
            {"type": i.type, "weight": i.weight, "severity": i.severity.value, "evidence": i.description}
            for i in risk.indicators
        ]
    )


evidence("INV-1049-metro-cleaning.md")

INV-1049: 0/100 → LOW



,type,weight,severity,evidence
0,CONFIRMED_MATCH,0,LOW,Bank account ****7733 matches the verified account on file for Metro Cleaning Services.
1,CONFIRMED_MATCH,0,LOW,"Bank account holder ""Metro Cleaning Services CC"" matches the supplier name."
2,CONFIRMED_MATCH,0,LOW,"Invoice contact domain ""metrocleaning.co.za"" matches the supplier record."
3,CONFIRMED_MATCH,0,LOW,"Amount R18,400 is within the expected range of R10,000-R20,000."


## 4. Why is INV-2005 (Nexus Advisory) CRITICAL?

In [5]:
evidence("INV-2005-nexus-advisory-CRITICAL.pdf")

NX-0917: 80/100 → CRITICAL



,type,weight,severity,evidence
0,SUPPLIER_NOT_VERIFIED,15,MEDIUM,"Nexus Advisory Partners is not a verified supplier, so there is no confirmed banking r..."
1,ACCOUNT_HOLDER_MISMATCH,25,HIGH,"Bank account holder ""K Mokoena"" does not match the supplier name ""Nexus Advisory Partn..."
2,PERSONAL_EMAIL_DOMAIN,15,HIGH,"Invoice contact k.mokoena.nexus@gmail.com uses a personal email provider (gmail.com), ..."
3,THRESHOLD_AVOIDANCE,15,MEDIUM,"Amount R98,500 sits just below the R100,000 dual-authorization threshold - a common pa..."
4,URGENCY_INDICATOR,10,MEDIUM,Payment is marked IMMEDIATE/urgent. POL-004 treats urgency as a social-engineering sig...


## 5. Edge cases the rules get right

**A recurring monthly invoice is not a duplicate.** INV-2001 (September) follows INV-1049 (August) from the
same supplier. Their line items name different months and the dates differ, so it's not flagged.
**INV-2004** re-bills the *August* service under a new number: identical line items, so it **is** flagged.

In [6]:
for name in ["INV-2001-metro-cleaning-LOW.pdf", "INV-2004-metro-cleaning-DUPLICATE.pdf"]:
    ctx, risk = results[name]
    dup = [i.description for i in risk.indicators if i.type == "DUPLICATE_INVOICE"]
    print(f"{ctx.invoice.id}: {dup[0] if dup else 'no duplicate found'}")

INV-2001: no duplicate found
INV-2004: Possible duplicate of invoice INV-1049 (R18,400, 2026-08-01) from the same supplier with the same billed items.


**An unverified supplier is onboarding, not a bank change.** Secure IT Solutions has no verified record,
so there is nothing to compare its bank account with: `SUPPLIER_NOT_VERIFIED`, never `BANK_DETAILS_CHANGED`.

In [7]:
ctx, risk = results["INV-1051-secure-it-solutions.md"]
print("Supplier verified:", ctx.supplier.verified)
print([i.type for i in risk.indicators])

Supplier verified: False
['SUPPLIER_NOT_VERIFIED', 'CONFIRMED_MATCH', 'AMOUNT_EXCEEDS_THRESHOLD']


## 6. Where the rules alone are not enough

INV-1050 changes Digital Print Co's bank account and says "Payment needed urgently… previous account has been
closed". The rules catch the bank change, but the score is only MEDIUM, because the pressure is in the
*wording* (priority is HIGH, not IMMEDIATE). That wording is what the AI review adds as `SOCIAL_ENGINEERING`.
A MEDIUM result also triggers the bounded deep-dive agent in phase 3.

In [8]:
evidence("INV-1050-digital-print-co.md")

INV-1050: 45/100 → MEDIUM



,type,weight,severity,evidence
0,BANK_DETAILS_CHANGED,30,CRITICAL,Invoice bank account ****6655 (FNB) does not match the verified account on file ****21...
1,CONFIRMED_MATCH,0,LOW,"Bank account holder ""Digital Print Co (Pty) Ltd"" matches the supplier name."
2,CONFIRMED_MATCH,0,LOW,"Invoice contact domain ""digitalprint.co.za"" matches the supplier record."
3,AMOUNT_EXCEEDS_THRESHOLD,15,MEDIUM,"Amount R195,500 exceeds the R100,000 threshold with no supplier-specific range on file..."


## 7. The minimum-action rule

The model may recommend something **more** cautious than this floor, never less. The floor is code.

In [9]:
from trustagent.domain import RiskLevel
from trustagent.rules.recommendation import resolve_recommended_action

pd.DataFrame(
    [
        {
            "level": lvl.value if lvl else "no score",
            "verified supplier": v,
            "minimum": minimum_action(lvl, v).value,
            "model proposes APPROVE →": resolve_recommended_action("APPROVE_PAYMENT", lvl, v)[0].value,
        }
        for lvl in [RiskLevel.CRITICAL, RiskLevel.HIGH, RiskLevel.MEDIUM, RiskLevel.LOW, None]
        for v in (True, False)
    ]
)

,level,verified supplier,minimum,model proposes APPROVE →
0,CRITICAL,True,HOLD_PAYMENT,HOLD_PAYMENT
1,CRITICAL,False,HOLD_PAYMENT,HOLD_PAYMENT
2,HIGH,True,HOLD_PAYMENT,HOLD_PAYMENT
3,HIGH,False,HOLD_PAYMENT,HOLD_PAYMENT
4,MEDIUM,True,REQUEST_VERIFICATION,REQUEST_VERIFICATION
5,MEDIUM,False,REQUEST_VERIFICATION,REQUEST_VERIFICATION
6,LOW,True,APPROVE_PAYMENT,APPROVE_PAYMENT
7,LOW,False,REQUEST_VERIFICATION,REQUEST_VERIFICATION
8,no score,True,ESCALATE,ESCALATE
9,no score,False,ESCALATE,ESCALATE


In [10]:
session.rollback()
session.close()
print("Rolled back: the demo database is unchanged.")

Rolled back: the demo database is unchanged.
